# Run the pyWOMBAT biogeochemical model in 1D water column configuration

In [1]:
import sys
import os
import itertools
import logging
from datetime import datetime
import numpy as np
import pandas as pd
import xarray as xr
import scipy as sci
from scipy.stats import qmc
import PyCO2SYS as pyco2
import matplotlib.pyplot as plt
import multiprocessing

# Ensure we are in the correct directory
os.chdir("/home/581/pjb581/py-WOMBAT/lite")
print(os.getcwd())
from main import main  # Import the main function from main.py

# Define output directory and ensure it exists (MAKE SURE A FORWARD SLASH EXISTS AT END)
OUTPUT_DIR = "/g/data/es60/pjb581/py-WOMBAT/output/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("python version =",sys.version[:5])
print("numpy version =", np.__version__)
print("xarray version =", xr.__version__)
print("scipy version =", sci.__version__)
print("PyCO2SYS version =", pyco2.__version__)

print(datetime.now())

"""
NOTE: Need read permissions for groups es60, gb6, qv56
"""

/home/581/pjb581/py-WOMBAT/lite
python version = 3.10.
numpy version = 1.24.4
xarray version = 2023.8.0
scipy version = 1.15.1
PyCO2SYS version = 1.8.3.4
2025-07-07 09:28:17.618449


'\nNOTE: Need read permissions for groups es60, gb6, qv56\n'

### Check git branch

In [2]:
!git branch

  master
* pyWOMBAT-on-Gadi
  pyWOMBAT-on-Gadi_microbes


## Set up the experiments

#### Here, you could add additional parameters to the "info" array
#### These parameters must be named the same as those that are input to the model in the "parameters.py" file in src/
#### If you add another parameter, this must be carried through to the call to main() in the cells below

In [3]:
%%time

year = 2001
days = 30
lons = [270.0] * 16
lats = [10.0, 5.0, 0.0, -5.0, -10., -15.0, -20.0, -25.0, -30.0, -35.0, -40.0, -45.0, -50.0, -55.0, -60.0, -65.0]
atm_co2 = 400.0

info = np.array([[1.0, year, days, lons[0]-360.0, lats[0], atm_co2],
                 [2.0, year, days, lons[1]-360.0, lats[1], atm_co2],
                 [3.0, year, days, lons[2]-360.0, lats[2], atm_co2],
                 [4.0, year, days, lons[3]-360.0, lats[3], atm_co2],
                 [5.0, year, days, lons[4]-360.0, lats[4], atm_co2],
                 [6.0, year, days, lons[5]-360.0, lats[5], atm_co2],
                 [7.0, year, days, lons[6]-360.0, lats[6], atm_co2],
                 [8.0, year, days, lons[7]-360.0, lats[7], atm_co2],
                 [9.0, year, days, lons[8]-360.0, lats[8], atm_co2],
                 [10.0, year, days, lons[9]-360.0, lats[9], atm_co2],
                 [11.0, year, days, lons[10]-360.0, lats[10], atm_co2],
                 [12.0, year, days, lons[11]-360.0, lats[11], atm_co2],
                 [13.0, year, days, lons[12]-360.0, lats[12], atm_co2],
                 [14.0, year, days, lons[13]-360.0, lats[13], atm_co2],
                 [15.0, year, days, lons[14]-360.0, lats[14], atm_co2],
                 [16.0, year, days, lons[15]-360.0, lats[15], atm_co2]])


CPU times: user 112 μs, sys: 0 ns, total: 112 μs
Wall time: 118 μs


### Make parameter set a pandas DataFrame

In [4]:
%%time

names = ["expnum", "year", "days", "longitude", "latitude", "atmCO2"]
paramsets = pd.DataFrame(info, columns=names)
paramsets


CPU times: user 1.66 ms, sys: 1.57 ms, total: 3.23 ms
Wall time: 3.06 ms


,expnum,year,days,longitude,latitude,atmCO2
0,1.0,2001.0,30.0,-90.0,10.0,400.0
1,2.0,2001.0,30.0,-90.0,5.0,400.0
2,3.0,2001.0,30.0,-90.0,0.0,400.0
3,4.0,2001.0,30.0,-90.0,-5.0,400.0
4,5.0,2001.0,30.0,-90.0,-10.0,400.0
5,6.0,2001.0,30.0,-90.0,-15.0,400.0
6,7.0,2001.0,30.0,-90.0,-20.0,400.0
7,8.0,2001.0,30.0,-90.0,-25.0,400.0
8,9.0,2001.0,30.0,-90.0,-30.0,400.0
9,10.0,2001.0,30.0,-90.0,-35.0,400.0


## Run the experiments

In [5]:
%%time

# setup the log file for each experiment
def setup_logging():
    log_file = f"experiment_output_{multiprocessing.current_process().pid}.log"
    logging.basicConfig(filename=log_file, level=logging.INFO, format="%(asctime)s - %(message)s")


# Define function to run one experiment
def run_experiment(exp):

    setup_logging()
    
    expnum, yr, dayl, lon, lat, atm_co2 = exp
    
    logging.info(f"\n🚀 Running Experiment with Params: {exp}")
    logging.info(f"Process {multiprocessing.current_process().pid} started at {datetime.now()}")
    
    # Run the main function with these parameters
    main(expnum, yr, dayl, lon, lat, atm_co2)

    logging.info(f"Process {multiprocessing.current_process().pid} finished at {datetime.now()}")
    logging.info(f"✅ Experiment Complete: {exp}")
    


CPU times: user 2 μs, sys: 2 μs, total: 4 μs
Wall time: 7.87 μs


### Cut out the experiments already run

This code looks at the output netcdf files to check if some experiments have already completed and will remove these from the list of experiments that are run in the next cell

In [6]:
%%time

done = set()
for fn in os.listdir(OUTPUT_DIR):
    if "exp" in fn and fn.endswith(".nc"):
            try:
                # Extract the experiment number
                exp_str = fn.split("exp")[-1].split(".nc")[0]
                expnum = int(exp_str)
                done.add(expnum)
            except ValueError:
                # In case conversion fails, skip this file
                continue

todo = [exp for exp in info if exp[0] not in done]
np.shape(todo)


CPU times: user 2.58 ms, sys: 0 ns, total: 2.58 ms
Wall time: 1.99 ms


(16, 6)

In [ ]:
%%time

# Run experiments in parallel using multiprocessing
if __name__ == "__main__":
    num_processes = min(min(multiprocessing.cpu_count(), 16), len(todo))  # Use at most the available cores to a max of 16
    print("Running %i parallel experiments"%(num_processes))
    with multiprocessing.Pool(processes=num_processes) as pool:
        pool.map(run_experiment, todo)

    # Check generated output files
    output_files = os.listdir("output")
    print("\nGenerated output files:", output_files)
    

Running 16 parallel experiments


/g/data/es60/pjb581/miniforge3/envs/pyWOMBAT_env/lib/python3.10/site-packages/numpy/core/fromnumeric.py:3464: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/g/data/es60/pjb581/miniforge3/envs/pyWOMBAT_env/lib/python3.10/site-packages/numpy/core/_methods.py:192: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
